In [2]:
# Make this notebook work from either fine-tuning/ or fine-tuning/pre-demo/
# (idempotent: re-running is safe)
import os
from pathlib import Path
_here = Path.cwd()
if _here.name in ('pre-demo', 'live-demo'):
    os.chdir(_here.parent)
print('cwd:', Path.cwd())


cwd: <repo>\fine-tuning


# Lab 00 · Synthetic Data Generation

A model is only as smart as its training data — and right now you have none, just a knowledge base in markdown. This lab turns that one file into a grounded training set: split `data/acme_health_kb.md` into sections, have `gpt-4.1-mini` write grounded Q&A pairs from each, and save them as train / validation JSONL in Azure's fine-tuning format. You walk away with 50–80 grounded pairs in `acme_training.jsonl` and `acme_validation.jsonl`, ready for Lab 01 — *the model just wrote its own training set.*

**Prereqs:** `pip install -r requirements.txt`, `.env` filled in, and `az login`.

---
## Step 1 — Load configuration

In [3]:
import base64
import json
import os
import random
from pathlib import Path

from azure.core.exceptions import ClientAuthenticationError
from azure.identity import AzureCliCredential, get_bearer_token_provider
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv()

AZURE_OPENAI_ENDPOINT = os.environ['AZURE_OPENAI_ENDPOINT']
AZURE_OPENAI_API_VERSION = os.environ.get('AZURE_OPENAI_API_VERSION', '2025-04-01-preview')
GENERATOR_DEPLOYMENT = os.environ.get('GENERATOR_DEPLOYMENT', 'gpt-4.1-mini')
TENANT_ID = os.environ.get('AZURE_TENANT_ID')

PAIRS_PER_SECTION = 8
TRAIN_SPLIT = 0.85
KB_FILE = Path('data/acme_health_kb.md')
TRAIN_FILE = Path('data/acme_training.jsonl')
VALID_FILE = Path('data/acme_validation.jsonl')
TOKEN_SCOPE = 'https://cognitiveservices.azure.com/.default'

SYSTEM_PROMPT = (
    'You are the Acme Health AI Assistant. Answer member '
    'questions accurately according to Acme Health\'s official policies, '
    'pharmacy procedures, plan benefits, and the Acme Health Portal portal.'
)

if not TENANT_ID:
    raise RuntimeError('AZURE_TENANT_ID is missing from fine-tuning/.env.')

credential = AzureCliCredential(tenant_id=TENANT_ID)
try:
    access_token = credential.get_token(TOKEN_SCOPE)
except ClientAuthenticationError as exc:
    raise RuntimeError(
        f'Azure CLI is not authenticated to resource tenant {TENANT_ID}. '
        f'Run `az login --tenant {TENANT_ID}`, select the expected subscription, '
        'restart the kernel, and rerun this cell.'
    ) from exc

payload = access_token.token.split('.')[1]
payload += '=' * (-len(payload) % 4)
token_tenant = json.loads(base64.urlsafe_b64decode(payload))['tid']
if token_tenant.lower() != TENANT_ID.lower():
    raise RuntimeError(
        f'Credential tenant {token_tenant} does not match AZURE_TENANT_ID {TENANT_ID}. '
        f'Run `az login --tenant {TENANT_ID}`, restart the kernel, and rerun this cell.'
    )

client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    azure_ad_token_provider=get_bearer_token_provider(credential, TOKEN_SCOPE),
    api_version=AZURE_OPENAI_API_VERSION,
)

print(f'Endpoint         : {AZURE_OPENAI_ENDPOINT}')
print(f'Authenticated tid: {token_tenant}')
print(f'Generator model  : {GENERATOR_DEPLOYMENT}')
print(f'Pairs per section: {PAIRS_PER_SECTION}')
print(f'Train split      : {int(TRAIN_SPLIT * 100)}%')

Endpoint         : https://<your-resource>.cognitiveservices.azure.com/
Authenticated tid: <your-tenant-id>
Generator model  : gpt-4.1-mini
Pairs per section: 8
Train split      : 85%


---
## Step 2 — Split the knowledge base into sections

Each `## ` heading in `acme_health_kb.md` becomes one chunk the generator
focuses on. Smaller chunks give the model less rope to hallucinate.

In [11]:
kb_text = KB_FILE.read_text(encoding='utf-8')
raw_sections = kb_text.split('\n## ')[1:]

sections = []
for block in raw_sections:
    text = ('## ' + block).strip()
    if len(text) < 200:
        continue
    title = text.splitlines()[0].lstrip('#').strip()
    sections.append({'title': title, 'content': text})

print(f'Loaded {len(sections)} substantive sections:')
for section in sections:
    print(f'   - {section["title"]}')

Loaded 7 substantive sections:
   - Pharmacy and Prescriptions
   - Appointments and Provider Access
   - Acme Health Plus Plans
   - ACME Online Portal
   - Billing and Insurance
   - Privacy, Identity, and Security
   - Support Channels


---
## Step 3 — Generate Q&A pairs from each section

Each section is rewritten into `PAIRS_PER_SECTION` question/answer pairs that
are strictly grounded in the source text — no hallucinated phone numbers,
copays, or policies.

In [12]:
GENERATION_PROMPT = '''You are a dataset curator creating training examples for the Acme Health Member Services AI assistant.

Given the knowledge base section below, generate exactly {n} distinct question-and-answer pairs.

Rules:
- Every answer must be grounded ONLY in the provided text. Do NOT add information not present.
- Mix question types: direct lookups, "what happens if..." scenarios, how-to questions, and natural caller phrasings.
- Phrase questions the way a Acme Health MEMBER would say them on a call (not corporate-speak).
- Answers should be warm, complete, and factually accurate — what a great ACME agent would say.
- Return ONLY a JSON array, no markdown fences, no commentary. Format:
[
  {{"question": "...", "answer": "..."}},
  ...
]

Knowledge base section:
---
{content}
---'''

MAX_GENERATION_ATTEMPTS = 3
all_examples = []
errors = []

for idx, section in enumerate(sections, 1):
    print(f'[{idx}/{len(sections)}] Generating for: {section["title"]}', end=' ')
    section_examples = []

    for attempt in range(1, MAX_GENERATION_ATTEMPTS + 1):
        try:
            response = client.chat.completions.create(
                model=GENERATOR_DEPLOYMENT,
                messages=[
                    {'role': 'system', 'content': 'You generate JSON training data. Return only valid JSON arrays.'},
                    {
                        'role': 'user',
                        'content': GENERATION_PROMPT.format(
                            n=PAIRS_PER_SECTION,
                            content=section['content'],
                        ),
                    },
                ],
                temperature=0.7,
                max_tokens=2000,
            )
            raw = response.choices[0].message.content.strip()
            if raw.startswith('```'):
                raw = raw.split('```')[1]
                if raw.startswith('json'):
                    raw = raw[4:]
                raw = raw.strip()

            pairs = json.loads(raw)
            if not isinstance(pairs, list):
                raise ValueError('The model response was not a JSON array.')

            section_examples = []
            for pair in pairs:
                if not isinstance(pair, dict):
                    continue
                question = pair.get('question', '').strip()
                answer = pair.get('answer', '').strip()
                if question and answer:
                    section_examples.append({
                        'messages': [
                            {'role': 'system', 'content': SYSTEM_PROMPT},
                            {'role': 'user', 'content': question},
                            {'role': 'assistant', 'content': answer},
                        ]
                    })

            if len(section_examples) != PAIRS_PER_SECTION:
                raise ValueError(
                    f'Expected {PAIRS_PER_SECTION} complete pairs, '
                    f'but received {len(section_examples)}.'
                )
            break
        except Exception as exc:
            message = str(exc)
            if 'does not match resource tenant' in message or 'Tenant provided in token' in message:
                raise RuntimeError(
                    f'Authentication stopped before processing more sections. Run '
                    f'`az login --tenant {TENANT_ID}`, then restart the kernel and rerun '
                    'Steps 1-3.'
                ) from exc
            if attempt < MAX_GENERATION_ATTEMPTS:
                print(f'(attempt {attempt} failed: {message}; retrying)', end=' ')
            else:
                print(f'ERROR after {attempt} attempts: {message}')
                errors.append({'section': section['title'], 'error': message})

    if section_examples and len(section_examples) == PAIRS_PER_SECTION:
        all_examples.extend(section_examples)
        print(f'-> {len(section_examples)} pairs')

print(f'\nTotal examples generated: {len(all_examples)}')
if errors:
    print(f'Sections with errors: {len(errors)}')
    for error in errors:
        print(f'   - {error["section"]}: {error["error"]}')

[1/7] Generating for: Pharmacy and Prescriptions -> 8 pairs
[2/7] Generating for: Appointments and Provider Access -> 8 pairs
[3/7] Generating for: Acme Health Plus Plans -> 8 pairs
[4/7] Generating for: ACME Online Portal -> 8 pairs
[5/7] Generating for: Billing and Insurance -> 8 pairs
[6/7] Generating for: Privacy, Identity, and Security -> 8 pairs
[7/7] Generating for: Support Channels -> 8 pairs

Total examples generated: 56


---
## Step 4 — Validate and deduplicate

In [8]:
valid_examples = []
seen = set()
dropped_invalid = 0
dropped_dupes = 0

for ex in all_examples:
    msgs = ex.get('messages', [])
    roles = [m.get('role') for m in msgs]
    if roles != ['system', 'user', 'assistant']:
        dropped_invalid += 1
        continue
    if any(not m.get('content', '').strip() for m in msgs):
        dropped_invalid += 1
        continue
    q = msgs[1]['content'].lower().strip()
    if q in seen:
        dropped_dupes += 1
        continue
    seen.add(q)
    valid_examples.append(ex)

print(f'Valid examples   : {len(valid_examples)}')
print(f'Dropped invalid  : {dropped_invalid}')
print(f'Dropped duplicate: {dropped_dupes}')

if len(valid_examples) < 10:
    print('\nWARNING: Azure OpenAI fine-tuning requires at least 10 examples.')
    print('Raise PAIRS_PER_SECTION in Step 1 and re-run Step 3.')

Valid examples   : 56
Dropped invalid  : 0
Dropped duplicate: 0


---
## Step 5 — Split, shuffle, save JSONL files

The writer uses plain UTF-8 JSONL: one complete JSON object per line, with no byte-order mark.

In [9]:
CANONICAL_TRAINING_EXAMPLES = [
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "When must I submit a ACME pharmacy refill to have it filled that business day?"},
            {"role": "assistant", "content": "Submit the refill before 2:00 PM Pacific for same-business-day filling. Requests after 2:00 PM are filled the next business day."},
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "What copay applies to a 90-day mail-order supply of a preferred generic?"},
            {"role": "assistant", "content": "A Tier 1 preferred generic has a $20 copay for a 90-day mail-order supply."},
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "How are members told about a major formulary update?"},
            {"role": "assistant", "content": "ACME emails affected members 60 days before a major formulary change."},
        ]
    },
]

canonical_questions = {
    row["messages"][1]["content"].strip().lower()
    for row in CANONICAL_TRAINING_EXAMPLES
}
generated_pool = [
    row for row in valid_examples
    if row["messages"][1]["content"].strip().lower() not in canonical_questions
]

random.seed(42)
random.shuffle(generated_pool)

target_train_size = int((len(generated_pool) + len(CANONICAL_TRAINING_EXAMPLES)) * TRAIN_SPLIT)
generated_train_size = max(0, target_train_size - len(CANONICAL_TRAINING_EXAMPLES))
train = CANONICAL_TRAINING_EXAMPLES + generated_pool[:generated_train_size]
valid = generated_pool[generated_train_size:]
random.shuffle(train)

def save_jsonl(rows, path):
    path.parent.mkdir(exist_ok=True)
    with open(path, 'w', encoding='utf-8', newline='\n') as file:
        for row in rows:
            file.write(json.dumps(row, ensure_ascii=False) + '\n')

save_jsonl(train, TRAIN_FILE)
save_jsonl(valid, VALID_FILE)

print(f'{TRAIN_FILE} - {len(train)} examples (includes 3 required policy facts)')
print(f'{VALID_FILE} - {len(valid)} examples')
print(f'\nTotal: {len(train) + len(valid)} examples ready for Lab 01.')

data\acme_training.jsonl - 50 examples (includes 3 required policy facts)
data\acme_validation.jsonl - 9 examples

Total: 59 examples ready for Lab 01.


---
## Step 6 — Preview a few examples

In [10]:
# Step 6 - Preview a few examples. Self-contained: if `train` isn't in
# memory (kernel restart or cells run out of order), reload it from the
# saved training file so this cell never NameErrors.
if 'train' not in globals():
    import json
    train = [json.loads(l) for l in open(TRAIN_FILE, encoding='utf-8-sig') if l.strip()]

for ex in train[:5]:
    msgs = ex['messages']
    user = next((m['content'] for m in msgs if m['role'] == 'user'), '')
    assistant = next((m['content'] for m in msgs if m['role'] == 'assistant'), '')
    print(f'Q: {user}')
    print(f'A: {assistant}')
    print('-' * 70)


Q: Is my call with you protected under any privacy laws?
A: Yes, all conversations with me are protected under HIPAA, which means your health information stays private and secure.
----------------------------------------------------------------------
Q: I have a billing question on Saturday. Who can I talk to?
A: Billing support is available Monday through Friday, 8 AM to 5 PM Pacific Time. Since it's Saturday, you might want to try ACME Online for help or call back during billing hours.
----------------------------------------------------------------------
Q: What copay applies to a 90-day mail-order supply of a preferred generic?
A: A Tier 1 preferred generic has a $20 copay for a 90-day mail-order supply.
----------------------------------------------------------------------
Q: What's the cost difference between a 30-day and 90-day supply for a preferred generic drug?
A: For a Tier 1 preferred generic medication, the 30-day copay is $10, while the 90-day mail order copay is $20, sav

---
## Done

Two files are now sitting in `data/`:

| File | Purpose |
|------|---------|
| `acme_training.jsonl`   | Training set for Lab 01 |
| `acme_validation.jsonl` | Validation set for Lab 01 |

Open **`01_supervised_fine_tuning.ipynb`** next.